# 05.00 章节概述：多模态机械臂综合实验（香橙派部署）

> ⚠️ 本章为**硬件端综合实验**，部署在香橙派 OrangePi AIPro + JAKA 机械臂上，**不在 CANNLab 上运行**。本章是前 4 章（YOLO 视觉 / 语音 OCR / LLM 微调 / VLA 训练）的**综合应用与真机落地**。

## 本章节定位

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">章节</th><th align="left">平台</th><th align="left">本章关联</th></tr>
<tr><td align="left">第 1 章 YOLO 目标检测</td><td align="left">CANNLab 云上</td><td align="left">本章复用：YOLO 模型转 OM 在昇腾 310 上推理</td></tr>
<tr><td align="left">第 2 章 语音 OCR</td><td align="left">CANNLab 云上</td><td align="left">本章复用：华为云 SIS 语音识别/合成</td></tr>
<tr><td align="left">第 3 章 LLM 微调</td><td align="left">CANNLab 云上</td><td align="left">本章复用：DeepSeek API 做意图解析</td></tr>
<tr><td align="left">第 4 章 VLA 训练</td><td align="left">CANNLab 云上</td><td align="left">本章关联：VLA 训练的 ACT 模型也可部署到香橙派</td></tr>
<tr><td align="left">**第 5 章 综合实验**</td><td align="left">**香橙派真机**</td><td align="left">**★ 把前 4 章的能力整合到真实机械臂**</td></tr>
</table>

## 系统架构

```
┌─────────────────────────────────────────────────────┐
│                 多模态控制中枢 (start.py)             │
├─────────────┬─────────────────┬─────────────────────┤
│  👁️ 视觉    │   🎤 语音        │   🧠 LLM 大模型      │
│  YOLO OM    │   华为云SIS      │   DeepSeek API      │
│  水果检测   │   ASR→文字       │   意图解析+多轮对话   │
│  (Ascend310)│   TTS←播报       │                     │
├─────────────┴─────────────────┴─────────────────────┤
│              意图调度（parse_intent）                 │
│   pick / detect / chat / calib / quit / status      │
├─────────────────────────────────────────────────────┤
│        机械臂执行层（JAKA SDK + 夹爪 PWM）           │
└─────────────────────────────────────────────────────┘
```

## 硬件环境

<table style="text-align: left; margin-left: 0;">
<tr><th align="left">硬件</th><th align="left">型号</th><th align="left">说明</th></tr>
<tr><td align="left">主控</td><td align="left">香橙派 OrangePi AIPro 20T</td><td align="left">昇腾 Ascend 310B4 NPU</td></tr>
<tr><td align="left">机械臂</td><td align="left">JAKA miniCOBO</td><td align="left">6 自由度协作臂，TCP 连接（IP `10.5.5.100`）</td></tr>
<tr><td align="left">摄像头</td><td align="left">USB 摄像头 ×2</td><td align="left">前置 + 腕部</td></tr>
<tr><td align="left">夹爪</td><td align="left">PWM 夹爪</td><td align="left">wiringPi GPIO 控制（Pin 19）</td></tr>
</table>

## 香橙派软件环境

<table style="text-align: left; margin-left: 0;">
<tr><th align="left">依赖</th><th align="left">版本</th><th align="left">说明</th></tr>
<tr><td align="left">OS</td><td align="left">Ubuntu 22.04</td><td align="left">香橙派预装</td></tr>
<tr><td align="left">CANN</td><td align="left">8.0.RC1+</td><td align="left">香橙派预装（NPU 加速）</td></tr>
<tr><td align="left">Python</td><td align="left">3.10</td><td align="left">香橙派默认</td></tr>
<tr><td align="left">PyTorch</td><td align="left">2.1+</td><td align="left">+ torch_npu</td></tr>
<tr><td align="left">acllite</td><td align="left">-</td><td align="left">昇腾轻量推理库（OM 模型）</td></tr>
<tr><td align="left">JAKA SDK</td><td align="left">jkrc.so</td><td align="left">机械臂控制（TCP 通信）</td></tr>
<tr><td align="left">wiringPi</td><td align="left">-</td><td align="left">GPIO 控制（夹爪 PWM）</td></tr>
</table>

> 💡 香橙派环境通常由教师/助教预先配置，学习者无需自行安装 CANN/torch_npu。

## 目录结构与项目说明

```
05_OrangePi-JAKAArm/
├── 05.00_chapter_intro.ipynb     ← 本文件（章节概述）
├── README.md                      详细说明文档
├── 0-StarterPack/                香橙派开机配置（教师用，播报IP等）
├── 1-Speech&LLMs/                语音+LLM 基础教学（百度/华为云/文心）
├── 2-YOLO/                       YOLO 推理教学（3种方式：ultralytics/onnxruntime/acllite）
├── 3-JAKAMinicobo/               JAKA 机械臂 SDK 教学样例（登录/移动/夹爪）
└── 4-FruitSorter/                ★ 综合主项目（多模态水果分拣）
    ├── start.py                  ← 主程序入口（多模态控制）
    ├── speech/                   语音（华为云SIS）+ LLM（DeepSeek）
    ├── perception/               YOLO 视觉检测（OM模型，Ascend 310）
    ├── control/                  JAKA 机械臂 + 夹爪控制
    ├── agent/                    任务调度
    ├── tools/                    手眼标定工具
    └── training/                 YOLO 训练+导出（可选，在CANNLab做）
```

## 项目文件作用与调用关系

### 核心调用链

```
start.py（主循环）
  ├── speech/huawei_voice.py    → ASR 录音识别 + TTS 语音播报
  ├── speech/llm_deepseek.py    → DeepSeek 意图解析 + 多轮对话
  ├── perception/yolo_detector.py → YOLO OM 模型推理（检测水果）
  ├── control/jaka_arm.py       → JAKA 机械臂运动控制
  ├── control/gripper_pwm.py    → 夹爪开合（GPIO PWM）
  └── agent/task_agent.py       → 任务调度（意图→动作映射）
```

### 各文件作用

<table style="text-align: left; margin-left: 0;">
<tr><th align="left">文件</th><th align="left">作用</th><th align="left">调用方式</th></tr>
<tr><td align="left">`start.py`</td><td align="left">主程序入口，多模态主循环</td><td align="left">`sudo python3 start.py`</td></tr>
<tr><td align="left">`speech/huawei_voice.py`</td><td align="left">华为云 SIS 语音（ASR/TTS）</td><td align="left">被 start.py 调用</td></tr>
<tr><td align="left">`speech/llm_deepseek.py`</td><td align="left">DeepSeek LLM（意图解析+对话）</td><td align="left">被 start.py 调用</td></tr>
<tr><td align="left">`perception/yolo_detector.py`</td><td align="left">YOLO OM 推理（Ascend 310）</td><td align="left">被 start.py 调用</td></tr>
<tr><td align="left">`control/jaka_arm.py`</td><td align="left">JAKA SDK 机械臂控制</td><td align="left">被 start.py 调用</td></tr>
<tr><td align="left">`control/gripper_pwm.py`</td><td align="left">夹爪 GPIO PWM</td><td align="left">被 start.py 调用</td></tr>
<tr><td align="left">`agent/task_agent.py`</td><td align="left">意图→任务调度</td><td align="left">被 start.py 调用</td></tr>
<tr><td align="left">`tools/eyehand_calib.py`</td><td align="left">手眼标定</td><td align="left">独立运行（部署前标定）</td></tr>
</table>

## 配置与运行

### 1. 配置凭证

在 `4-FruitSorter/` 目录下创建 `.env`（参考 `.env.example`）：

```ini
HUAWEI_SIS_AK=你的AccessKey
HUAWEI_SIS_SK=你的SecretKey
HUAWEI_SIS_REGION=cn-east-3
HUAWEI_SIS_PROJECT_ID=你的项目ID
DEEPSEEK_API_KEY=你的Key
```

> 💡 三个模块都可缺失降级：无华为凭证→跳过语音；无 DeepSeek→退化为关键词规则；无麦克风→键盘输入。

### 2. 运行

```bash
cd 4-FruitSorter
sudo python3 start.py    # sudo 是因为夹爪 GPIO 需要 root
```

### 3. 三种控制方式

<table style="text-align: left; margin-left: 0;">
<tr><th align="left">方式</th><th align="left">触发</th><th align="left">示例</th></tr>
<tr><td align="left">👁️ 视觉</td><td align="left">说"检测"/"抓芒果"</td><td align="left">YOLO 识别水果并自动抓取</td></tr>
<tr><td align="left">🎤 语音</td><td align="left">自然语言指令</td><td align="left">"把柠檬放到右边"</td></tr>
<tr><td align="left">🧠 LLM</td><td align="left">复杂对话</td><td align="left">"你能做什么？" → DeepSeek 理解回复</td></tr>
</table>

> 💡 详细的各子模块说明和教学步骤，见各子目录的 README.md。